## Fix the broken schakels in N roads
## create shp files for arcgis

In [4]:
import geopandas as gpd
import pandas as pd
from pathlib import Path

# Input and output paths
input_path = Path(r"P:\bovenregionale-stresstest-hwn\Analysis\Merged_Outputs\Aggregated_schakels_merged_with_area_edited_ver04.gpkg")
output_path = Path(r"P:\bovenregionale-stresstest-hwn\Analysis\Merged_Outputs\Aggregated_schakels_merged_with_area_edited_ver04_edited.gpkg")

# Read the input GeoPackage
gdf_src = gpd.read_file(input_path)

# Define the pairs to merge
pairs = [
    ("033-1030-L", "033-1030-R"),
    ("035-0006-L", "035-0006-R"),
    ("048-1010-L", "048-1010-R"),
    ("035-0002-L", "035-0002-R"),
    ("057-0020-L", "057-0020-R"),
    ("057-0030-L", "057-0030-R"),
    ("057-0050-L", "057-0050-R"),
    ("057-0060-L", "057-0060-R"),
    ("057-0070-L", "057-0070-R"),
    ("057-0080-L", "057-0080-R"),
    ("059-0010-L", "059-0010-R"),
    ("059-0030-L", "059-0030-R"),
    ("059-0060-L", "059-0060-R"),
    ("059-0040-L", "059-0040-R"),
    ("059-0050-L", "059-0050-R"),
    ("059-0020-L", "059-0020-R"),
    ("035-0004-L", "035-0004-R"),
    ("015-0096-L", "015-0096-R"),

]

# Columns to sum
sum_cols = ["total_length", "flooded_length", "bridge_length_sum", "tunnel_length_sum", "total_damage"]

# Column to take max
max_col = "F_EV2_ma_max"

merged_rows = []
merged_ids = set()

for left, right in pairs:
    row_L = gdf_src[gdf_src["NETWERKSCH_HWN"] == left]
    row_R = gdf_src[gdf_src["NETWERKSCH_HWN"] == right]

    if row_L.empty or row_R.empty:
        continue

    row_L = row_L.iloc[0]
    row_R = row_R.iloc[0]

    merged_data = row_R.copy()

    # Sum numeric columns
    for col in sum_cols:
        merged_data[col] = row_L[col] + row_R[col]

    # Max for F_EV2_ma_max
    merged_data[max_col] = max(row_L[max_col], row_R[max_col])

    # Compute fraction_flooded
    merged_data["fraction_flooded"] = (
        merged_data["flooded_length"] / merged_data["total_length"]
        if merged_data["total_length"] != 0 else 0
    )

    merged_rows.append(merged_data)

    # Track merged IDs
    merged_ids.update([left, right])

# Create GeoDataFrame for merged rows
gdf_merged_pairs = gpd.GeoDataFrame(merged_rows, crs=gdf_src.crs)

# Keep all other rows that were not merged
gdf_rest = gdf_src[~gdf_src["NETWERKSCH_HWN"].isin(merged_ids)]

# Combine merged pairs with the rest
gdf_final = gpd.GeoDataFrame(pd.concat([gdf_merged_pairs, gdf_rest], ignore_index=True), crs=gdf_src.crs)

# Save to GeoPackage
gdf_final.to_file(output_path, driver="GPKG")

print(f"Final dataset saved with {len(gdf_final)} rows to {output_path}")

CPLE_AppDefinedError: b"sqlite3_exec(UPDATE gpkg_contents SET last_change = strftime('%Y-%m-%dT%H:%M:%fZ','now') WHERE lower(table_name) = lower('Aggregated_schakels_merged_with_area_edited_ver04_edited')) failed: unable to open database file"

Exception ignored in: 'fiona._shim.gdal_flush_cache'
Traceback (most recent call last):
  File "fiona\_err.pyx", line 201, in fiona._err.GDALErrCtxManager.__exit__
fiona._err.CPLE_AppDefinedError: b"sqlite3_exec(UPDATE gpkg_contents SET last_change = strftime('%Y-%m-%dT%H:%M:%fZ','now') WHERE lower(table_name) = lower('Aggregated_schakels_merged_with_area_edited_ver04_edited')) failed: unable to open database file"


Final dataset saved with 468 rows to P:\bovenregionale-stresstest-hwn\Analysis\Merged_Outputs\Aggregated_schakels_merged_with_area_edited_ver04_edited.gpkg


In [ ]:
area_gpkg = Path(r"P:\bovenregionale-stresstest-hwn\Analysis\Areas.gpkg")
gdf_area = gpd.read_file(area_gpkg)

AFR_path  = Path(r"P:\bovenregionale-stresstest-hwn\Analysis\Final_outputs\AFR_Points.shp")
OPR_path = Path(r"P:\bovenregionale-stresstest-hwn\Analysis\Final_outputs\OPR_Points.shp")

OPR = gpd.read_file(OPR_path)
AFR = gpd.read_file(AFR_path)



In [2]:
import geopandas as gpd
from pathlib import Path

# Load data
area_gpkg = Path(r"P:\bovenregionale-stresstest-hwn\Analysis\Areas.gpkg")
gdf_area = gpd.read_file(area_gpkg)

AFR_path = Path(r"P:\bovenregionale-stresstest-hwn\Analysis\Final_outputs\AFR_Points.shp")
OPR_path = Path(r"P:\bovenregionale-stresstest-hwn\Analysis\Final_outputs\OPR_Points.shp")

OPR = gpd.read_file(OPR_path)
AFR = gpd.read_file(AFR_path)

# Check the column name for area name in gdf_area
print(gdf_area.columns)  # Look for something like 'area_name'

# Perform spatial join
OPR_with_area = gpd.sjoin(OPR, gdf_area[['geometry', 'name']], how='left', predicate='within')
AFR_with_area = gpd.sjoin(AFR, gdf_area[['geometry', 'name']], how='left', predicate='within')

# Save updated files
OPR_with_area.to_file(OPR_path, driver='ESRI Shapefile')
AFR_with_area.to_file(AFR_path, driver='ESRI Shapefile')

Index(['gml_id', 'nationalCo', 'localId', 'namespace', 'nationalLe',
       'national_1', 'country', 'name', 'geometry'],
      dtype='object')


C:\Users\gunaratn\AppData\Local\Temp\ipykernel_11916\535882985.py:22: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  OPR_with_area.to_file(OPR_path, driver='ESRI Shapefile')
C:\Users\gunaratn\AppData\Local\Temp\ipykernel_11916\535882985.py:23: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  AFR_with_area.to_file(AFR_path, driver='ESRI Shapefile')
